In [2]:
import csv
import pandas as pd
from scipy import stats
from scipy.stats import levene, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print("Đã import thành công các thư viện cần thiết!")


Đã import thành công các thư viện cần thiết!


In [5]:
import os

file_name = 'BT2.225092301_insurance_survey.csv'
if not os.path.exists(file_name) and os.path.exists(f'BTTH/2/BT2.{file_name}'):
    file_name = f'BTTH/2/BT2.{file_name}'

df = pd.read_csv(file_name)

print("Dữ liệu 5 dòng đầu tiên:")
display(df.head())
print("\nThông tin tổng quan bộ dữ liệu:")
df.info()


Dữ liệu 5 dòng đầu tiên:


,Age,Gender,Education,Marital Status,Years Employed,Satisfaction*,Premium/Deductible**
0,36,F,Some college,Divorced,4,4,N
1,55,F,Some college,Divorced,2,1,N
2,61,M,Graduate degree,Widowed,26,3,N
3,65,F,Some college,Married,9,4,N
4,53,F,Graduate degree,Married,6,4,N



Thông tin tổng quan bộ dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Age                   24 non-null     int64 
 1   Gender                24 non-null     object
 2   Education             24 non-null     object
 3   Marital Status        24 non-null     object
 4   Years Employed        24 non-null     int64 
 5   Satisfaction*         24 non-null     int64 
 6   Premium/Deductible**  24 non-null     object
dtypes: int64(3), object(4)
memory usage: 1.4+ KB


In [7]:
education_groups = {edu: group for edu, group in df.groupby('Education')}

print(f"Tổng số nhóm tìm thấy: {len(education_groups)}\n")
for edu_name, group_df in education_groups.items():
    print(f"- Nhóm trình độ '{edu_name}': {len(group_df)} mẫu")


Tổng số nhóm tìm thấy: 3

- Nhóm trình độ 'College graduate': 9 mẫu
- Nhóm trình độ 'Graduate degree': 8 mẫu
- Nhóm trình độ 'Some college': 7 mẫu


In [ ]:
target_col = 'Satisfaction*' if 'Satisfaction*' in df.columns else 'Satisfaction'
alpha = 0.05

print("=" * 65)
print("BƯỚC 1: ĐẶT GIẢ THUYẾT BÀI TOÁN (KIỂM ĐỊNH LEVENE)")
print("  - H0 (Giả thuyết không): Phương sai về mức độ hài lòng giữa các nhóm")
print("                          trình độ học vấn (Education) là đồng nhất.")
print("  - H1 (Giả thuyết đối)  : Có ít nhất một nhóm có phương sai khác biệt")
print("                          (phương sai không đồng nhất).")
print("=" * 65)

samples = [group[target_col] for group in education_groups.values()]

stat_levene, p_value_levene = levene(*samples)

print(f"\nBƯỚC 2: KẾT QUẢ KIỂM ĐỊNH LEVENE")
print(f"  - Giá trị thống kê (Statistic): {stat_levene:.4f}")
print(f"  - p-value                     : {p_value_levene:.4f}")
print(f"  - Mức ý nghĩa alpha (α)       : {alpha}")

print("\nBƯỚC 3: KẾT LUẬN")
if p_value_levene > alpha:
    print(f"  => Vì p-value ({p_value_levene:.4f}) > {alpha}, ta CHẤP NHẬN giả thuyết H0.")
    print("     Kết luận: Phương sai về mức độ hài lòng giữa các nhóm trình độ học vấn là ĐỒNG NHẤT.")
    print("     (Dữ liệu thỏa mãn điều kiện phương sai đồng nhất để thực hiện ANOVA one-way).")
else:
    print(f"  => Vì p-value ({p_value_levene:.4f}) <= {alpha}, ta BÁC BỎ giả thuyết H0.")
    print("     Kết luận: Phương sai giữa các nhóm không đồng nhất.")


BƯỚC 1: ĐẶT GIẢ THUYẾT BÀI TOÁN (KIỂM ĐỊNH LEVENE)
  - H0 (Giả thuyết không): Phương sai về mức độ hài lòng giữa các nhóm
                          trình độ học vấn (Education) là đồng nhất.
  - H1 (Giả thuyết đối)  : Có ít nhất một nhóm có phương sai khác biệt
                          (phương sai không đồng nhất).

BƯỚC 2: KẾT QUẢ KIỂM ĐỊNH LEVENE
  - Giá trị thống kê (Statistic): 0.2652
  - p-value                     : 0.7696
  - Mức ý nghĩa alpha (α)       : 0.05

BƯỚC 3: KẾT LUẬN
  => Vì p-value (0.7696) > 0.05, ta CHẤP NHẬN giả thuyết H0.
     Kết luận: Phương sai về mức độ hài lòng giữa các nhóm trình độ học vấn là ĐỒNG NHẤT.
     (Dữ liệu thỏa mãn điều kiện phương sai đồng nhất để thực hiện ANOVA one-way).


In [10]:
print("=" * 65)
print("BƯỚC 1: ĐẶT GIẢ THUYẾT BÀI TOÁN (KIỂM ĐỊNH ANOVA ONE-WAY)")
print("  - H0 (Giả thuyết không): Giá trị trung bình mức độ hài lòng của các nhóm")
print("                          trình độ học vấn là như nhau (μ1 = μ2 = μ3).")
print("  - H1 (Giả thuyết đối)  : Có ít nhất một nhóm có giá trị trung bình mức độ")
print("                          hài lòng khác với các nhóm còn lại.")
print("=" * 65)

stat_anova, p_value_anova = f_oneway(*samples)

print(f"\nBƯỚC 2: KẾT QUẢ KIỂM ĐỊNH ANOVA ONE-WAY")
print(f"  - Giá trị F-statistic  : {stat_anova:.4f}")
print(f"  - p-value              : {p_value_anova:.4f}")
print(f"  - Mức ý nghĩa alpha (α): {alpha}")

print("\nBƯỚC 3: KẾT LUẬN")
if p_value_anova < alpha:
    print(f"  => Vì p-value ({p_value_anova:.4f}) < {alpha}, ta BÁC BỎ giả thuyết H0.")
    print("     Kết luận: Có sự KHÁC BIỆT CÓ Ý NGHĨA THỐNG KÊ về mức độ hài lòng trung bình")
    print("     giữa các nhóm trình độ học vấn khác nhau.")
else:
    print(f"  => Vì p-value ({p_value_anova:.4f}) >= {alpha}, ta CHƯA ĐỦ BẰNG CHỨNG bác bỏ H0.")
    print("     Kết luận: Không có sự khác biệt có ý nghĩa thống kê về mức độ hài lòng trung bình giữa các nhóm.")


BƯỚC 1: ĐẶT GIẢ THUYẾT BÀI TOÁN (KIỂM ĐỊNH ANOVA ONE-WAY)
  - H0 (Giả thuyết không): Giá trị trung bình mức độ hài lòng của các nhóm
                          trình độ học vấn là như nhau (μ1 = μ2 = μ3).
  - H1 (Giả thuyết đối)  : Có ít nhất một nhóm có giá trị trung bình mức độ
                          hài lòng khác với các nhóm còn lại.

BƯỚC 2: KẾT QUẢ KIỂM ĐỊNH ANOVA ONE-WAY
  - Giá trị F-statistic  : 3.9247
  - p-value              : 0.0356
  - Mức ý nghĩa alpha (α): 0.05

BƯỚC 3: KẾT LUẬN
  => Vì p-value (0.0356) < 0.05, ta BÁC BỎ giả thuyết H0.
     Kết luận: Có sự KHÁC BIỆT CÓ Ý NGHĨA THỐNG KÊ về mức độ hài lòng trung bình
     giữa các nhóm trình độ học vấn khác nhau.


In [11]:
# Kiểm tra điều kiện p-value từ ANOVA
if p_value_anova < alpha:
    print("=" * 70)
    print("VÌ p-value CỦA ANOVA < 0.05, TIẾN HÀNH KIỂM ĐỊNH SÂU TUKEY HSD:")
    print("=" * 70)
    
    # Thực hiện Tukey HSD test
    tukey = pairwise_tukeyhsd(endog=df[target_col], groups=df['Education'], alpha=alpha)
    
    print("\nBẢNG KẾT QUẢ KIỂM ĐỊNH TUKEY HSD:")
    print(tukey)
    
    print("\n" + "=" * 70)
    print("KẾT LUẬN CHI TIẾT CÁC CẶP SO SÁNH:")
    print("=" * 70)
    for row in tukey.summary()[1:]:
        g1, g2, meandiff, p_adj, lower, upper, reject = [str(val).strip() for val in row]
        if reject == 'True':
            print(f"• Cặp [{g1}] vs [{g2}]:")
            print(f"   - Chênh lệch trung bình: {meandiff}")
            print(f"   - p-adj = {p_adj} < {alpha} (reject = True)")
            print(f"   => CÓ sự khác biệt có ý nghĩa thống kê về mức độ hài lòng.\n")
        else:
            print(f"• Cặp [{g1}] vs [{g2}]:")
            print(f"   - Chênh lệch trung bình: {meandiff}")
            print(f"   - p-adj = {p_adj} >= {alpha} (reject = False)")
            print(f"   => KHÔNG có sự khác biệt có ý nghĩa thống kê về mức độ hài lòng.\n")
else:
    print(f"p-value của kiểm định ANOVA là {p_value_anova:.4f} >= {alpha}, không cần thực hiện kiểm định sâu Tukey HSD.")


VÌ p-value CỦA ANOVA < 0.05, TIẾN HÀNH KIỂM ĐỊNH SÂU TUKEY HSD:

BẢNG KẾT QUẢ KIỂM ĐỊNH TUKEY HSD:
          Multiple Comparison of Means - Tukey HSD, FWER=0.05          
     group1           group2     meandiff p-adj   lower   upper  reject
-----------------------------------------------------------------------
College graduate Graduate degree   1.0556 0.1003 -0.1715  2.2826  False
College graduate    Some college  -0.3016 0.8231 -1.5742  0.9711  False
 Graduate degree    Some college  -1.3571 0.0409 -2.6641 -0.0502   True
-----------------------------------------------------------------------

KẾT LUẬN CHI TIẾT CÁC CẶP SO SÁNH:
• Cặp [College graduate] vs [Graduate degree]:
   - Chênh lệch trung bình: 1.0556
   - p-adj = 0.1003 >= 0.05 (reject = False)
   => KHÔNG có sự khác biệt có ý nghĩa thống kê về mức độ hài lòng.

• Cặp [College graduate] vs [Some college]:
   - Chênh lệch trung bình: -0.3016
   - p-adj = 0.8231 >= 0.05 (reject = False)
   => KHÔNG có sự khác biệt có ý nghĩa t